# Kimee: put the final weights into the repo (Phase 7, once)

The host wants code, config **and weights** in one repo. GitHub refuses files > 100 MB, so
`src/weights_io.py` strips each checkpoint to the model weights only (no optimizer state) and splits it into
90 MB parts plus `manifest.json` (size + SHA-256).

Models = exp101: `x101_2048_m28` + `r50_2048` (checkpoints are only **read** from `solar_filament/output/detectron2/`).
Output: `solar_filament/Phase 7/weights/` (new folder). **CPU runtime is enough** (~5 min).

Afterwards: download the folder `Phase 7/weights/` and put its files into the repo folder `weights/` on the laptop,
then tell Claude (it checks them and you commit + push).

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
CKPT = {n: f"{DRIVE_DIR}/output/detectron2/{n}/model_final.pth" for n in ("x101_2048_m28", "r50_2048")}
OUT = f"{DRIVE_DIR}/Phase 7/weights"
for n, c in CKPT.items():
    assert os.path.exists(c), f"Checkpoint not found: {c}"
    print(n, f"{os.path.getsize(c) / 1e6:.0f} MB")

In [ ]:
import os
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

In [ ]:
os.makedirs(OUT, exist_ok=True)
for n, c in CKPT.items():
    !python src/weights_io.py pack --ckpt "{c}" --name {n} --out "{OUT}"
!python src/weights_io.py unpack --dir "{OUT}"
# the joined .pth files are only a check; remove them so only parts + manifest are downloaded
for n in CKPT:
    os.remove(f"{OUT}/{n}.pth")
print(sorted(os.listdir(OUT)))
!cat "{OUT}/manifest.json"